In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_KEY_HERE"

In [2]:
!pip install langchain langchain-core langchain_community langchain_huggingface faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 kB 6.5 MB/s eta

In [7]:
from langchain_community.vectorstores import FAISS ,Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
chunks = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [8]:
model = HuggingFaceEmbeddings(model="Qwen/Qwen3-Embedding-0.6B")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=model,
    collection_name='exp'
)

In [9]:
retriver = vector_store.as_retriever(search_kwargs={"k":2})

In [10]:
query = "what is langchain databases"

results = retriver.invoke(query)

In [11]:
for r in results:
  print(r)

page_content='Chroma is a vector database optimized for LLM-based search.'
page_content='LangChain helps developers build LLM applications easily.'


## MMR

In [12]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [40]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=model,
)

In [59]:
retriver = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k":3,"lambda_unit":0.7}
)

In [60]:
query = "what is langchain"
results = retriver.invoke(query)

In [61]:
for r in results:
  print(r.page_content)

LangChain makes it easy to work with LLMs.
Embeddings are vector representations of text.
LangChain supports Chroma, FAISS, Pinecone, and more.


## multi query retriver

In [20]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [67]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever

llm = HuggingFaceEndpoint(repo_id="deepseek-ai/DeepSeek-V3.2-Exp")
ai = ChatHuggingFace(llm=llm)

vector_store = Chroma.from_documents(
    documents=all_docs,
    embedding=model,
    collection_name="exp3"
)

simple_retriver = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':5,"lambda_unit":0.3}
)

multi_retriver = MultiQueryRetriever.from_llm(
    retriever=simple_retriver,
    llm=ai
)

results = multi_retriver.invoke("How to improve energy levels and maintain balance?")

for r in results:
  print(r)

page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.' metadata={'source': 'H2'}
page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.' metadata={'source': 'H5'}


## contextual compression Reteriver